## Step 1 — Join actual values with predictions

In [0]:
from pyspark.sql import functions as F

# Use the Gold test table from Notebook 32
TEST_GOLD_TABLE = ("predictivemaintenancecatalog.gold_data.streaming")

PREDICTION_TABLE = ("predictivemaintenancecatalog.gold_data.test_failure_predictions")

# Actual outcomes
actual_df = (
    spark.table(TEST_GOLD_TABLE)
    .select(
        "UDI",
        F.col("Machine_failure").cast("int").alias("actual_failure")
    )
)

# Model predictions
predicted_df = (
    spark.table(PREDICTION_TABLE)
    .select(
        "UDI",
        "failure_probability",
        "predicted_failure"
    )
)

# Match predictions with actual outcomes
evaluation_df = actual_df.join(
    predicted_df,
    on="UDI",
    how="inner"
)

print("Actual observations:", actual_df.count())
print("Predictions:", predicted_df.count())
print("Matched observations:", evaluation_df.count())

display(evaluation_df.limit(10))

## Step 2 — Calculate test performance

In [0]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    fbeta_score
)

# 2,000 rows: small enough for pandas
pdf = evaluation_df.select(
    "actual_failure",
    "predicted_failure"
).toPandas()

y_true = pdf["actual_failure"]
y_pred = pdf["predicted_failure"]

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

print(classification_report(
    y_true, y_pred,
    labels=[0, 1],
    zero_division=0
))

print("F2 Score:", fbeta_score(
    y_true, y_pred,
    beta=2,
    zero_division=0
))

print("True Positve:", tp)
print("False Negatives:", fn)
print("False Positives:", fp)
print("True Negative:", tn)
print("Alert Rate:", (tp + fp) / len(y_true))